<a href="https://colab.research.google.com/github/SahilRamdoss/Real_Time_Capuchin_Detector/blob/Colab/Real_Time_Capuchin_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Connecting Github Repository and downloading dependencies

In [ ]:
!git clone https://github.com/SahilRamdoss/Real_Time_Capuchin_Detector.git

Cloning into 'Real_Time_Capuchin_Detector'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 106 (delta 38), reused 90 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (106/106), 9.85 MiB | 12.77 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [ ]:
!pip install tensorflow==2.21.0 numpy==2.3.5 scipy pyyaml audiomentations librosa

# Create Data pipeline

Creating a list of the required file paths

In [ ]:
from glob import glob

CAPUCHIN_FILES_PATHS = glob(r"/content/drive/MyDrive/Datasets/data/Parsed_Capuchinbird_Clips/*.wav")
NON_CAPUCHINE_FILE_PATHS = glob(r"/content/drive/MyDrive/Datasets/data/Parsed_Not_Capuchinbird_Clips/*.wav")

print(f"Number of Capuchin files: {len(CAPUCHIN_FILES_PATHS)}")
print(f"Number of Non-Capuchin files: {len(NON_CAPUCHINE_FILE_PATHS)}")

Number of Capuchin files: 217
Number of Non-Capuchin files: 593


We are downsampling the majority class here to prevent class imbalance.

In [ ]:
import random

random.shuffle(NON_CAPUCHINE_FILE_PATHS)

NON_CAPUCHINE_FILE_PATHS = NON_CAPUCHINE_FILE_PATHS[:len(CAPUCHIN_FILES_PATHS)]

print(f"Number of Capuchin files: {len(CAPUCHIN_FILES_PATHS)}")
print(f"Number of Non-Capuchin files: {len(NON_CAPUCHINE_FILE_PATHS)}")

Number of Capuchin files: 217
Number of Non-Capuchin files: 217


Add the project root in system paths

In [ ]:
import os, sys

PROJECT_ROOT = "/content/Real_Time_Capuchin_Detector"

if PROJECT_ROOT not in sys.path:
  sys.path.append(PROJECT_ROOT)

Splitting the data into train, val and test datasets

In [ ]:
import random

TOTAL_FILES = (len(CAPUCHIN_FILES_PATHS) + len(NON_CAPUCHINE_FILE_PATHS))

TRAIN_SIZE = int(0.80 * TOTAL_FILES)

VAL_SIZE = (TOTAL_FILES - TRAIN_SIZE) // 2

LABELLED_CAPUCHIN_FILE_PATHS = [(path, 1) for path in CAPUCHIN_FILES_PATHS]

LABELLED_NON_CAPUCHIN_FILE_PATHS = [(path, 0) for path in NON_CAPUCHINE_FILE_PATHS]

LABELLED_COMBINED_FILE_PATHS = LABELLED_CAPUCHIN_FILE_PATHS + LABELLED_NON_CAPUCHIN_FILE_PATHS
random.shuffle(LABELLED_COMBINED_FILE_PATHS)

LABELLED_TRAIN_FILE_PATHS = LABELLED_COMBINED_FILE_PATHS[:TRAIN_SIZE]
LABELLED_VAL_FILE_PATHS = LABELLED_COMBINED_FILE_PATHS[TRAIN_SIZE: (TRAIN_SIZE + VAL_SIZE)]
LABELLED_TEST_FILE_PATHS = LABELLED_COMBINED_FILE_PATHS[(TRAIN_SIZE + VAL_SIZE):]

print(f"Size of train dataset: {len(LABELLED_TRAIN_FILE_PATHS)}")
print(f"Size of validation dataset: {len(LABELLED_VAL_FILE_PATHS)}")
print(f"Size of test dataset: {len(LABELLED_TEST_FILE_PATHS)}")

print("\n")

print(f"Train Dataset: {LABELLED_TRAIN_FILE_PATHS}")
print(f"Validation Dataset: {LABELLED_VAL_FILE_PATHS}")
print(f"Test Dataset: {LABELLED_TEST_FILE_PATHS}")

Size of train dataset: 347
Size of validation dataset: 43
Size of test dataset: 44


Train Dataset: [('/content/drive/MyDrive/Datasets/data/Parsed_Not_Capuchinbird_Clips/screech-owl-sounds-at-night-1.wav', 0), ('/content/drive/MyDrive/Datasets/data/Parsed_Capuchinbird_Clips/XC307385-0.wav', 1), ('/content/drive/MyDrive/Datasets/data/Parsed_Capuchinbird_Clips/XC395129-2.wav', 1), ('/content/drive/MyDrive/Datasets/data/Parsed_Capuchinbird_Clips/XC22397-1.wav', 1), ('/content/drive/MyDrive/Datasets/data/Parsed_Not_Capuchinbird_Clips/dove-bird-sounds-1.wav', 0), ('/content/drive/MyDrive/Datasets/data/Parsed_Capuchinbird_Clips/XC216012-14.wav', 1), ('/content/drive/MyDrive/Datasets/data/Parsed_Not_Capuchinbird_Clips/crickets-sound-effect-3.wav', 0), ('/content/drive/MyDrive/Datasets/data/Parsed_Not_Capuchinbird_Clips/crickets-chirping-noise-21.wav', 0), ('/content/drive/MyDrive/Datasets/data/Parsed_Capuchinbird_Clips/XC513083-1.wav', 1), ('/content/drive/MyDrive/Datasets/data/Parsed_Not_Cap

Simple check to see how balanced train dataset is

In [ ]:
capuchin_count = 0

for path, label in LABELLED_TRAIN_FILE_PATHS:
  if label == 1:
    capuchin_count += 1

print(f"Number of Capuchin calls in train dataset: {capuchin_count}")
print(f"Number of Non-capuchin calls in train dataset: {len(LABELLED_TRAIN_FILE_PATHS) - capuchin_count}")

Number of Capuchin calls in train dataset: 173
Number of Non-capuchin calls in train dataset: 174


Create data input pipeline

In [ ]:
from typing import Tuple
import numpy as np
import tensorflow as tf
import yaml
from training.data_prep.audio_normalisation import AudioNorm
from training.data_prep.audio_augmentation import AudioAug

#################################################
# Create instance of classes only once
#################################################

audio_norm = AudioNorm.from_config()
audio_aug = AudioAug

#################################################
# Functions to be used in data pipelines
#################################################

@tf.py_function(Tout=[tf.float32, tf.int32, tf.int32])
def get_audio_data(file_path:str, label: int) -> Tuple[np.ndarray, int, int]:
  """
  This function takes in the file path of the sound file and the corresponding label. It then return the audio samples, its sampling rate and the label.

  Params:
    file_path(str): File path of audio file
    label(int): Label for audio file (0 for Not Capuchin and 1 for Capuchin)

  Returns:
    The audio samples, sampling rate and label in the shape (audio samples, sampling rate, label)
  """

  raw = tf.io.read_file(file_path)
  audio, sr = tf.audio.decode_wav(raw, desired_channels=1)
  audio = tf.squeeze(audio, axis=-1).numpy()
  sr = sr.numpy()

  return audio, sr, label

@tf.py_function(Tout=[tf.float32, tf.int32, tf.int32])
def padding(audio: np.ndarray, sr: int, label: int) -> Tuple[np.ndarray, int, int]:
  audio, sr = audio_norm.silence_padding(audio, sr)
  return audio, sr, label

@tf.py_function(Tout=[tf.float32, tf.int32, tf.int32])
def random_clipping(audio: np.ndarray, sr: int, label: int) -> Tuple[np.ndarray, int, int]:
  audio, sr = audio_norm.random_clipping(audio, sr)
  return audio, sr, label

@tf.py_function(Tout=[tf.float32, tf.int32, tf.int32])
def audio_augmentation(audio: np.ndarray, sr: int, label: int) -> Tuple[np.ndarray, int, int]:
  audio = audio_aug.augment_audio(audio)
  return audio, sr, label


#######################################################
# Main code
#######################################################

train_paths, train_labels = zip(*LABELLED_TRAIN_FILE_PATHS)



training_dataset = (
    tf.data.Dataset.from_tensor_slices((list(train_paths), list(train_labels)))
    # Get audio data and sampling rate
    .map(get_audio_data, num_parallel_calls=tf.data.AUTOTUNE)
    # Apply padding if needed
    .map(padding, num_parallel_calls=tf.data.AUTOTUNE)
    # Apply the random clipping
    .map(random_clipping, num_parallel_calls=tf.data.AUTOTUNE)
    # Apply the audio augmentation
    .map(audio_augmentation, num_parallel_calls=tf.data.AUTOTUNE)
)